# ROMS-DART Data Assimilation Experiment: Further Assessment

## 1. Import Python libraries

In [ ]:
import pydartdiags.obs_sequence.obs_sequence as obsq

import os
import cmocean
import numpy             as np
import pandas            as pd
import xarray            as xr
import plotly.io         as pio
import cartopy.crs       as ccrs
import plotly.express    as px
import cartopy.feature   as cfeature
import matplotlib.pyplot as plt
import matplotlib.dates  as mdates

from pathlib              import Path
from IPython.display      import Markdown, display
from matplotlib.colors    import LogNorm
from pydartdiags.stats    import stats
from pydartdiags.matplots import matplots as mp

## 2. Define Paths and Variables

In [ ]:
# Path to DART repo (directory) 
basedir = Path(f"/glade/derecho/scratch/{os.environ['USER']}/inacawo/DART_training/models/ROMS_rutgers/work")

# Path to the obs_seq and obs_diag_output.nc files
mohadir = Path(f"/glade/derecho/scratch/gharamti/inacawo/DART_training/models/ROMS_rutgers/work")

# Roms restart template 
template = basedir / 'roms_template.nc'

# Observation Sequence
obs_seq_file  = mohadir / 'filter_output_eval' / 'obs_seq.final'  

print(f"obs_seq file: {obs_seq_file}")

# Make sure the obs_Seq file exists
assert obs_seq_file.exists(), 'obs_seq file not found'

run_dirs = {
    'exp1': basedir / 'filter_output_300_no' , 
    'exp2': mohadir / 'filter_output'        ,
    'exp3': mohadir / 'filter_output_300_250', 
    'exp4': mohadir / 'filter_output_300_100', 
    'exp5': mohadir / 'filter_output_eval'
}

files = {
    'increment' : 'increment.nc'    ,
    'prior_mean': 'preassim_mean.nc', 
    'post_mean' : 'output_mean.nc'  , 
    'prior_sd'  : 'preassim_sd.nc'  ,
    'post_sd'   : 'output_sd.nc'    , 
    'inflate'   : 'output_priorinf_mean.nc'
}

state = {
    'T':   {'name': 'temp', 'grid': 'rho', 'unit': 'C'},
    'S':   {'name': 'salt', 'grid': 'rho', 'unit': 'psu'},
    'SSH': {'name': 'zeta', 'grid': 'rho', 'unit': 'm'},
    'U':   {'name': 'u',    'grid': 'u',   'unit': 'm/s'},   
    'V':   {'name': 'v',    'grid': 'v',   'unit': 'm/s'}
}

field     = ['T', 'S', 'U', 'V', 'SSH']
Variables = ['Temperature', 'Salinity', 'U Current', 'V Current', 'Sea Surface Height'] 

roms = xr.open_dataset(template)

# Functions & Tools
datasets = {
    exp: {
        file_key: xr.open_dataset(path / fname)
        for file_key, fname in files.items()
    }
    for exp, path in run_dirs.items()
}

def roms_z_rho(ds, grid="rho"):
    """
    Compute ROMS z_rho depths for Vtransform=2.
    Returns z with same horizontal grid as requested.
    """

    s  = ds["s_rho"]
    Cs = ds["Cs_r"]
    hc = ds["hc"]
    h  = ds["h"]

    zeta = ds["zeta"].squeeze() if "zeta" in ds else 0.0

    # move h/zeta to u or v grid if needed
    if grid == "u":
        h = 0.5 * (h.isel(xi_rho=slice(0, -1)) + h.isel(xi_rho=slice(1, None)))
        zeta = 0.5 * (
            zeta.isel(xi_rho=slice(0, -1)) +
            zeta.isel(xi_rho=slice(1, None))
        )
        h = h.rename({"xi_rho": "xi_u"})
        zeta = zeta.rename({"xi_rho": "xi_u"})

    elif grid == "v":
        h = 0.5 * (h.isel(eta_rho=slice(0, -1)) + h.isel(eta_rho=slice(1, None)))
        zeta = 0.5 * (
            zeta.isel(eta_rho=slice(0, -1)) +
            zeta.isel(eta_rho=slice(1, None))
        )
        h = h.rename({"eta_rho": "eta_v"})
        zeta = zeta.rename({"eta_rho": "eta_v"})

    # ROMS Vtransform = 2
    z0 = (hc * s + Cs * h) / (hc + h)
    z  = zeta + (zeta + h) * z0

    return z

def extract_profile(ds, roms, var_key, xi, eta):
    vname = state[var_key]["name"]
    grid  = state[var_key]["grid"]

    if grid == "rho":
        data = ds[vname].isel(xi_rho=xi, eta_rho=eta).squeeze()
        z = roms_z_rho(roms, grid="rho")
        depth = -z.isel(xi_rho=xi, eta_rho=eta).squeeze()

    elif grid == "u":
        data = ds[vname].isel(xi_u=xi, eta_u=eta).squeeze()
        z = roms_z_rho(roms, grid="u")
        depth = -z.isel(xi_u=xi, eta_u=eta).squeeze()

    elif grid == "v":
        data = ds[vname].isel(xi_v=xi, eta_v=eta).squeeze()
        z = roms_z_rho(roms, grid="v")
        depth = -z.isel(xi_v=xi, eta_v=eta).squeeze()

    return data, depth

def top_diff_locations(diff_map, n=5):
    stacked = diff_map.stack(points=diff_map.dims)
    stacked = stacked.where(np.isfinite(stacked), drop=True)

    top = stacked.sortby(stacked, ascending=False).isel(points=slice(0, n))

    locs = []
    for p in top.points.values:
        eta, xi = p
        locs.append((int(xi), int(eta), float(top.sel(points=p))))

    return locs

def get_lon_lat(roms, grid, xi, eta):
    if grid == "rho":
        lon = roms["lon_rho"].isel(xi_rho=xi, eta_rho=eta).values
        lat = roms["lat_rho"].isel(xi_rho=xi, eta_rho=eta).values

    elif grid == "u":
        lon = roms["lon_u"].isel(xi_u=xi, eta_u=eta).values
        lat = roms["lat_u"].isel(xi_u=xi, eta_u=eta).values

    elif grid == "v":
        lon = roms["lon_v"].isel(xi_v=xi, eta_v=eta).values
        lat = roms["lat_v"].isel(xi_v=xi, eta_v=eta).values

    return float(lon), float(lat)

## A1. Additional Assessment: Vertical Localization 

In [ ]:
# Vertical Localization: Compare profiles
var   = "S"
stage = "increment"
vname = state[var]["name"]
grid  = state[var]["grid"]

xind = 1558 #1558
yind = 410  #406

X1, dep1 = extract_profile(datasets["exp1"][stage], roms, var, xind, yind)
X2, dep2 = extract_profile(datasets["exp2"][stage], roms, var, xind, yind)
X3, dep3 = extract_profile(datasets["exp3"][stage], roms, var, xind, yind)
X4, dep4 = extract_profile(datasets["exp4"][stage], roms, var, xind, yind)

lon, lat = get_lon_lat(roms, grid, xind, yind)

plt.figure(figsize=(6, 8))

plt.plot(X1, dep1, "-k" , label="No Vertical Loc")
plt.plot(X2, dep2, "--b", label="Vert Loc: 500m")
plt.plot(X3, dep3, "-.g", label="Vert Loc: 250m")
plt.plot(X4, dep4, ":r" , label="Vert Loc: 100m")

plt.gca().invert_yaxis()
plt.grid(True, linestyle="--", linewidth=0.5, alpha=0.5)

plt.xlabel(f'Increment ({state[var]["unit"]})', fontsize=14)
plt.ylabel("Depth (m)", fontsize=14)
plt.title(f"{var} profile at lon={lon:.2f}, lat={lat:.2f}", fontsize=15)

plt.legend(title=stage, title_fontsize=12)

plt.savefig('vert_loc.png', dpi=300, bbox_inches='tight')
plt.show()

display(Markdown(
"""### Assessment of Vertical Localization Sensitivity

This figure illustrates the sensitivity of the salinity analysis increment to the choice of vertical localization radius 
while maintaining a horizontal localization 0-weight distance of 300 km. Several key behaviors emerge:

- **No vertical localization**: produces a broad positive increment extending from the surface down to ~350 m.
- **500 m**: reduces the amplitude somewhat but still allows substantial deep influence.
- **250 m**: further confines the adjustment and substantially weakens the deep response.
- **100 m**: strongly suppresses influence below ~150–200 m and even changes the sign of the near-surface increment.

Overall, the vertical extent of the increment steadily shrinks as the localization radius decreases. 
It should be noted that this assessment is based on a single profile location. 
We may still need to test with different vertical localization distances to further optimize the setup.  
"""))

## A2. Additional Assessment: Withheld Observations 

In [ ]:
# Read the obs seq file into a DataFrame
obs = obsq.ObsSequence(obs_seq_file)
df  = obs.df.copy() 

this = df[df["type"] == "FLOAT_TEMPERATURE"].copy()

# Sort and keep upper 100 m
this = this[this["vertical"] <= 500].copy()
this = this.sort_values("vertical")

# Group by approximate float location
this["profile_id"] = (
    this["longitude"].round(2).astype(str)
    + ", "
    + this["latitude"].round(2).astype(str)
)

profiles = sorted(this["profile_id"].unique())

fig, axes = plt.subplots(1, len(profiles), figsize=(6 * len(profiles), 6))

if len(profiles) == 1:
    axes = [axes]

for ax, profile in zip(axes, profiles):

    prof = this[this["profile_id"] == profile].sort_values("vertical")

    depth = prof["vertical"].values
    obs   = prof["observation"].values
    prior = prof["prior_ensemble_mean"].values
    post  = prof["posterior_ensemble_mean"].values

    prior_rmse = np.sqrt(np.nanmean((obs - prior)**2))
    post_rmse  = np.sqrt(np.nanmean((obs - post )**2))

    ax.plot(obs,   depth, "ok", ms=6, label="Observation")
    ax.plot(prior, depth, "-b", lw=2, label=f"Prior RMSE={prior_rmse:.2f}")
    ax.plot(post,  depth, "-r", lw=2, label=f"Posterior RMSE={post_rmse:.2f}")

    if ax == axes[0]:
        ax.set_ylim(100, 0)
    elif ax == axes[1]:
        ax.set_ylim(500, 0)
        
    ax.set_xlabel("C", fontsize=13)
    ax.set_title(f"Float at {profile}", fontsize=12)
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=12)

axes[0].set_ylabel("Depth (m)", fontsize=13)

plt.suptitle("Withheld Float Temperature Verification", fontsize=16)
plt.tight_layout()

# plt.savefig('float_withheld.png', dpi=300, bbox_inches='tight')
plt.show()
roms.close()